# Jet Lag: The Game — Seattle Early-Game Strategy Notebook

**Goal**: Compute *optimal early-game questions* (yes/no splits) that most efficiently bisect the possibility space and identify which public transit station the hider is at.

**Scope** (per game start near Seattle Art Museum):
- Seattle city limits (N/S boundaries)
- All of Bellevue, Redmond, and Mercer Island
- Only these modes: **Link light rail** stops, **Seattle Monorail** stops, **King County Metro RapidRide** stops

**Why these stops?** They are discrete, named, public locations with published schedules. The hider must be at one of them (or very close) when "caught" by a question.

**Game rules reference**: [lifack.ch — Jet Lag: The Game rules & question types](https://www.lifack.ch/)

Key question families for early game:
- Axis-aligned (N/S/E/W of a latitude/longitude)
- Radius from a landmark (within X km of Y)
- Attribute (on a particular line/color, in a zone, etc.)
- More advanced: Tentacles, Radar/Matching, Measuring (used once possibility set is small)

This notebook focuses on **foundation**: clean candidate list + great interactive maps + landmark audit readiness.


## Quick Start

From project root:

```bash
uv sync
uv run jupyter lab
```

Open `notebooks/seattle_strategy.ipynb`, **Restart & Run All**.

All heavy data (GTFS, boundaries) is cached locally after first run.


In [1]:
import pathlib
import json
import zipfile
from collections import defaultdict
from functools import lru_cache

import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import osmnx as ox
from shapely.geometry import Point
from shapely.ops import unary_union
import requests

# ============== GAME CONSTANTS ==============
START = (47.6075, -122.3380)          # Near Seattle Art Museum (game start)
START_NAME = "Seattle Art Museum (approx)"

# Project data dir (run jupyter from repo root so cwd == project root)
DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(exist_ok=True)

# GTFS sources (robust primary + fallbacks)
KCM_GTFS_URLS = [
    "https://www.soundtransit.org/GTFS-KCM/google_transit.zip",
    "https://kingcounty.gov/metro/gtfs/google_transit.zip",
]
ST_RAIL_GTFS_URL = "https://www.soundtransit.org/GTFS-rail/40_gtfs.zip"

# Mode filters
RAPIDRIDE_SHORT_NAMES = ["C Line", "D Line", "E Line", "F Line", "G Line", "H Line"]
LINK_PATTERNS = ["1 Line", "2 Line"]

# Monorail is tiny — hardcode the two platforms (authoritative coords)
MONORAIL_STOPS = [
    {"stop_name": "Westlake Center", "stop_lat": 47.61205, "stop_lon": -122.33696},
    {"stop_name": "Seattle Center",  "stop_lat": 47.62126, "stop_lon": -122.34996},
]

print("Constants loaded. Python env ready.")
print(f"START point: {START_NAME} @ {START}")


Constants loaded. Python env ready.
START point: Seattle Art Museum (approx) @ (47.6075, -122.338)


## 1. Acquiring Latitude/Longitude for Candidate Stations

**Primary source: GTFS feeds** (best choice)
- Link light rail → Sound Transit official rail GTFS
- RapidRide buses → King County Metro GTFS (via Sound Transit mirror for reliability)
- These contain *exact* stop latitudes/longitudes used by the agencies and Google/Apple Maps.

**Alternative sources explored**
- Overpass API (OpenStreetMap `public_transport=station` or `highway=bus_stop`) — useful for landmarks/POIs later, but *not* authoritative for the exact set of RapidRide/Link stops a player would use.
- Nominatim (used via osmnx) — for city boundaries and landmark geocoding.
- OneBusAway / agency real-time APIs — overkill for static locations; require keys.

We therefore use **GTFS** as the single source of truth for candidate (lat, lon). Monorail is hardcoded (only two stops, rarely in bus GTFS).


In [2]:
# === Robust GTFS downloader (idempotent, cached, User-Agent) ===
def download_gtfs(urls, dest_dir: pathlib.Path) -> pathlib.Path:
    if isinstance(urls, str):
        urls = [urls]
    dest_dir.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": "JetLag-Seattle-Strategy-Notebook/1.0 (+https://github.com/james)"}
    for url in urls:
        try:
            zip_name = url.split("/")[-1]
            extract_dir = dest_dir / zip_name.replace(".zip", "")
            if extract_dir.exists() and any(p.is_file() for p in extract_dir.iterdir()):
                print(f"  Using cached {extract_dir.name}")
                return extract_dir
            print(f"  Downloading {zip_name} ...")
            r = requests.get(url, headers=headers, timeout=120)
            r.raise_for_status()
            (dest_dir / zip_name).write_bytes(r.content)
            with zipfile.ZipFile(dest_dir / zip_name) as zf:
                zf.extractall(extract_dir)
            print(f"  Extracted to {extract_dir}")
            return extract_dir
        except Exception as e:
            print(f"  {url} failed: {e}")
            continue
    raise RuntimeError(f"All sources failed for {urls}")


In [3]:
# === Focused loaders (no geometry, just points + mode flags) ===

def load_rapidride_stops(gtfs_dir: pathlib.Path) -> pd.DataFrame:
    routes = pd.read_csv(gtfs_dir / "routes.txt")
    trips = pd.read_csv(gtfs_dir / "trips.txt")
    stop_times = pd.read_csv(gtfs_dir / "stop_times.txt")
    stops = pd.read_csv(gtfs_dir / "stops.txt")

    rr_routes = routes[routes["route_short_name"].isin(RAPIDRIDE_SHORT_NAMES)]
    rr_trip_ids = trips[trips["route_id"].isin(rr_routes["route_id"])]["trip_id"]
    rr_stop_ids = stop_times[stop_times["trip_id"].isin(rr_trip_ids)]["stop_id"].unique()

    df = stops[stops["stop_id"].isin(rr_stop_ids)].copy()
    df["is_rapidride"] = True
    df["is_link"] = False
    df["is_monorail"] = False
    return df[["stop_name", "stop_lat", "stop_lon", "is_rapidride", "is_link", "is_monorail"]].drop_duplicates()


def load_link_stops(gtfs_dir: pathlib.Path) -> pd.DataFrame:
    routes = pd.read_csv(gtfs_dir / "routes.txt")
    trips = pd.read_csv(gtfs_dir / "trips.txt")
    stop_times = pd.read_csv(gtfs_dir / "stop_times.txt")
    stops = pd.read_csv(gtfs_dir / "stops.txt")

    # Match both short_name variants and long_name fallbacks
    mask = (
        routes["route_short_name"].str.contains("|".join(LINK_PATTERNS), na=False, regex=True) |
        routes["route_long_name"].str.contains("1 Line|2 Line|Lynnwood|Federal Way", na=False, case=False, regex=True)
    )
    link_routes = routes[mask]
    link_trip_ids = trips[trips["route_id"].isin(link_routes["route_id"])]["trip_id"]
    link_stop_ids = stop_times[stop_times["trip_id"].isin(link_trip_ids)]["stop_id"].unique()

    df = stops[stops["stop_id"].isin(link_stop_ids)].copy()
    df["is_rapidride"] = False
    df["is_link"] = True
    df["is_monorail"] = False
    return df[["stop_name", "stop_lat", "stop_lon", "is_rapidride", "is_link", "is_monorail"]].drop_duplicates()


def load_monorail_stops() -> pd.DataFrame:
    df = pd.DataFrame(MONORAIL_STOPS)
    df["is_rapidride"] = False
    df["is_link"] = False
    df["is_monorail"] = True
    return df[["stop_name", "stop_lat", "stop_lon", "is_rapidride", "is_link", "is_monorail"]]


def dedup_candidates(df_list) -> pd.DataFrame:
    """Round coords to ~1.1 m and collapse duplicate platforms/entrances."""
    all_df = pd.concat([d for d in df_list if len(d) > 0], ignore_index=True)
    all_df["_rlat"] = all_df["stop_lat"].round(5)
    all_df["_rlon"] = all_df["stop_lon"].round(5)
    grouped = (all_df
        .groupby(["_rlat", "_rlon"])
        .agg({
            "stop_name": "first",
            "stop_lat": "first",
            "stop_lon": "first",
            "is_rapidride": "max",
            "is_link": "max",
            "is_monorail": "max",
        })
        .reset_index(drop=True)
    )
    return grouped[["stop_name", "stop_lat", "stop_lon", "is_rapidride", "is_link", "is_monorail"]].copy()


In [4]:
# === Geographic scope filter (Seattle city limits + Bellevue + Redmond + Mercer Island) ===
@lru_cache(maxsize=1)
def get_scope_polygon():
    print("Fetching city boundary polygons via osmnx/Nominatim ...")
    cities = [
        "Seattle, Washington, USA",
        "Bellevue, Washington, USA",
        "Redmond, Washington, USA",
        "Mercer Island, Washington, USA",
    ]
    gdf = ox.geocode_to_gdf(cities)
    union = unary_union(gdf.geometry.tolist())
    print(f"  Union of {len(gdf)} city polygons ready.")
    return union

def filter_to_scope(df: pd.DataFrame) -> pd.DataFrame:
    poly = get_scope_polygon()
    mask = df.apply(lambda r: Point(r["stop_lon"], r["stop_lat"]).within(poly), axis=1)
    out = df[mask].reset_index(drop=True)
    return out

# === ¼-mile walkshed (union of buffers around every candidate) ===
def compute_walkshed(df: pd.DataFrame, radius_m: float = 402.336) -> "shapely.geometry.base.BaseGeometry":
    """Return the union of ¼-mile (≈402 m) buffers around all points.

    Uses EPSG:3857 (Web Mercator) for accurate meter-based buffering,
    then converts the result back to EPSG:4326 for folium.
    """
    if len(df) == 0:
        from shapely.geometry import Point as _Point
        return _Point(0, 0).buffer(0)

    pts = gpd.points_from_xy(df["stop_lon"], df["stop_lat"])
    gdf = gpd.GeoDataFrame(geometry=pts, crs="EPSG:4326")

    # Accurate buffering in meters
    proj = gdf.to_crs(3857)
    buf = proj.buffer(radius_m)

    # Union (works on GeoSeries in geopandas >=0.10 / shapely 2+)
    if hasattr(buf, "union_all"):
        union_proj = buf.union_all()
    else:
        union_proj = unary_union(buf)

    # Back to WGS84
    union_4326 = gpd.GeoSeries([union_proj], crs=3857).to_crs(4326).iloc[0]
    return union_4326


In [5]:
# === One-call pipeline (uses local data if present, downloads once) ===
def load_candidates() -> pd.DataFrame:
    # Link (Sound Transit rail feed)
    rail_dir = DATA_DIR / "40_gtfs"
    if not (rail_dir / "stops.txt").exists():
        rail_dir = download_gtfs(ST_RAIL_GTFS_URL, DATA_DIR)
    link_df = load_link_stops(rail_dir)

    # RapidRide (KCM feed)
    kcm_dir = DATA_DIR / "google_transit"
    if not (kcm_dir / "stops.txt").exists():
        kcm_dir = download_gtfs(KCM_GTFS_URLS, DATA_DIR)
    rr_df = load_rapidride_stops(kcm_dir)

    mono_df = load_monorail_stops()

    raw = dedup_candidates([link_df, rr_df, mono_df])
    print(f"Raw deduplicated locations (all feeds): {len(raw)}")

    scoped = filter_to_scope(raw)
    print(f"Final candidates inside scope: {len(scoped)}")
    print("   Breakdown:")
    print(f"     Link light rail : {int(scoped['is_link'].sum()):3d}")
    print(f"     RapidRide       : {int(scoped['is_rapidride'].sum()):3d}")
    print(f"     Monorail        : {int(scoped['is_monorail'].sum()):3d}")
    return scoped

candidates = load_candidates()
candidates.head(8)


Raw deduplicated locations (all feeds): 403
Fetching city boundary polygons via osmnx/Nominatim ...
  Union of 4 city polygons ready.
Final candidates inside scope: 263
   Breakdown:
     Link light rail :  71
     RapidRide       : 192
     Monorail        :   2


,stop_name,stop_lat,stop_lon,is_rapidride,is_link,is_monorail
0,SW Roxbury St & 20th Ave SW,47.517296,-122.358818,True,False,False
1,SW Roxbury St & 26th Ave SW,47.517429,-122.365547,True,False,False
2,SW Roxbury St & 20th Ave SW,47.517429,-122.358925,True,False,False
3,SW Roxbury St & 28th Ave SW,47.517456,-122.368050,True,False,False
4,35th Ave SW & SW Roxbury St,47.517731,-122.376778,True,False,False
5,SW Wildwood Pl & 46th Ave SW,47.520935,-122.390892,True,False,False
6,26th Ave SW & SW Barton St - Bay 2,47.520939,-122.365768,True,False,False
7,SW Wildwood Pl & 45th Ave SW,47.521000,-122.390617,True,False,False


## 2. Visualization — Stations on an Interactive Map of Seattle

We use **folium** (Leaflet.js) for clean, zoomable, filterable maps directly inside Jupyter.

Design choices:
- Full-resolution (no clustering) — every candidate is visible and clickable.
- Color coding by mode: **blue = Link**, **orange = RapidRide**, **green = Monorail**.
- Black marker = game start location.
- Popups show name + mode flags.
- CartoDB Positron tiles for a calm, high-contrast base (easy to read hundreds of points).
- LayerControl so you can toggle modes on/off while exploring.

This map is the foundation for the **Landmark Audit** section that will follow (you will visually validate POIs used in radius/tentacle questions).

**New:** The light blue shaded regions show the **union of all ¼-mile (≈400 m) buffers** around every candidate stop. This is the practical "walkshed" — anywhere the hider could reasonably reach on foot after alighting. Toggle the layer on/off via the control in the top-right.


In [6]:
def create_map(df: pd.DataFrame, center=START, zoom=11, walkshed=None):
    """Create an interactive folium map of candidates + start point.

    walkshed: optional shapely geometry (union of ¼-mile buffers). If provided,
    it is rendered as a semi-transparent toggleable layer underneath the markers.
    """
    m = folium.Map(
        location=center,
        zoom_start=zoom,
        tiles="cartodbpositron",
        control_scale=True,
        prefer_canvas=True,   # smoother with hundreds of markers
    )

    # Optional ¼-mile walkshed — render first so it sits under the markers
    if walkshed is not None:
        walkshed_group = folium.FeatureGroup(name="¼-mile walkshed", show=True)
        try:
            # shapely objects support the __geo_interface__ protocol
            gj = folium.GeoJson(
                data=walkshed,
                style_function=lambda f: {
                    "fillColor": "#5dade2",
                    "color": "#2980b9",
                    "weight": 0.7,
                    "fillOpacity": 0.22,
                },
                tooltip="Area within ¼ mile (~400 m) walk of any candidate stop",
            )
            gj.add_to(walkshed_group)
        except Exception:
            # Robust fallback
            ws_gs = gpd.GeoSeries([walkshed], crs="EPSG:4326")
            gj = folium.GeoJson(
                data=ws_gs.to_json(),
                style_function=lambda f: {
                    "fillColor": "#5dade2",
                    "color": "#2980b9",
                    "weight": 0.7,
                    "fillOpacity": 0.22,
                },
            )
            gj.add_to(walkshed_group)
        walkshed_group.add_to(m)

    # Game start
    folium.Marker(
        location=center,
        popup=f"<b>{START_NAME}</b><br>Game start location",
        icon=folium.Icon(color="black", icon="info-sign"),
    ).add_to(m)

    # Per-mode feature groups (for layer control)
    groups = {
        "Link (light rail)": folium.FeatureGroup(name="Link (light rail)", show=True),
        "RapidRide": folium.FeatureGroup(name="RapidRide", show=True),
        "Monorail": folium.FeatureGroup(name="Monorail", show=True),
    }

    color_map = {
        "Link (light rail)": "blue",
        "RapidRide": "#ff7f00",   # vivid orange
        "Monorail": "green",
    }

    for _, row in df.iterrows():
        mode = ("Link (light rail)" if row["is_link"]
                else "Monorail" if row["is_monorail"]
                else "RapidRide")
        folium.CircleMarker(
            location=(row["stop_lat"], row["stop_lon"]),
            radius=3.5,
            color=color_map[mode],
            fill=True,
            fill_opacity=0.85,
            weight=1.0,
            popup=folium.Popup(
                f"<b>{row['stop_name']}</b><br>"
                f"Link: {bool(row['is_link'])} | "
                f"RapidRide: {bool(row['is_rapidride'])} | "
                f"Monorail: {bool(row['is_monorail'])}",
                max_width=280,
            ),
        ).add_to(groups[mode])

    for g in groups.values():
        g.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    # Nice title box
    title_html = '''
        <div style="position: fixed; bottom: 12px; left: 12px; 
                    background-color: rgba(255,255,255,0.92); 
                    padding: 6px 10px; border-radius: 4px; 
                    box-shadow: 0 1px 4px rgba(0,0,0,0.2); z-index: 9999; font-size: 13px;">
            <b>Jet Lag: The Game</b> — Seattle + Mercer Island candidates<br>
            <span style="color:blue">●</span> Link &nbsp;
            <span style="color:#ff7f00">●</span> RapidRide &nbsp;
            <span style="color:green">●</span> Monorail<br>
            <span style="color:#5dade2">■</span> ¼-mile walkshed (toggle in layer control)
        </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    return m

# Compute the shaded walkshed once, then pass it to the map
walkshed = compute_walkshed(candidates)
m = create_map(candidates, walkshed=walkshed)
m   # Jupyter will render the interactive map


In [7]:
# Quick stats for the current possibility set
print("=== Candidate Set Summary ===")
print(f"Total stations: {len(candidates)}")
print()
print("Modes (mutually exclusive after dedup):")
for col in ["is_link", "is_rapidride", "is_monorail"]:
    print(f"  {col:13s}: {int(candidates[col].sum()):3d}")

print()
print("Geographic extent (lat/lon):")
print(f"  lat: {candidates['stop_lat'].min():.4f} ... {candidates['stop_lat'].max():.4f}")
print(f"  lon: {candidates['stop_lon'].min():.4f} ... {candidates['stop_lon'].max():.4f}")

# Rough "center of mass"
centroid_lat = candidates["stop_lat"].mean()
centroid_lon = candidates["stop_lon"].mean()
print(f"\nCentroid (mean): ({centroid_lat:.4f}, {centroid_lon:.4f})")
dist_km = (abs(centroid_lat - START[0])**2 + abs(centroid_lon - START[1])**2) ** 0.5 * 111.0
print(f"Distance from game start: {dist_km:.1f} km (approx)")

# Show a few Link stations (they are high-value early anchors)
print("\nSample Link stops in set:")
print(candidates[candidates["is_link"]][["stop_name", "stop_lat", "stop_lon"]].head(6).to_string(index=False))


=== Candidate Set Summary ===
Total stations: 263

Modes (mutually exclusive after dedup):
  is_link      :  71
  is_rapidride : 192
  is_monorail  :   2

Geographic extent (lat/lon):
  lat: 47.5173 ... 47.7336
  lon: -122.3931 ... -122.1091

Centroid (mean): (47.6116, -122.3305)
Distance from game start: 0.9 km (approx)

Sample Link stops in set:
    stop_name  stop_lat    stop_lon
Rainier Beach 47.522266 -122.279587
Rainier Beach 47.522953 -122.279083
      Othello 47.537529 -122.281471
      Othello 47.538502 -122.281693
Columbia City 47.559025 -122.292389
Columbia City 47.560280 -122.292892


## Next Steps (Roadmap)

1. **Landmark Audit** (highest value next)
   - Curated, user-editable table of strong POIs (Space Needle, Columbia Center, UW, Bellevue Downtown Park, Marymoor, etc.)
   - Visual audit map that overlays radius circles + the candidate points
   - Distance matrix pre-computed for fast "within X km of Y?" evaluation

2. **Question Engine**
   - Axis-aligned splits (choose best lat or lon threshold by information gain / balance)
   - Radius questions from audited landmarks (optimal radius per landmark)
   - Attribute questions (which lines pass through current possibility set?)
   - Automatic ranking of the top-5 questions by expected entropy reduction

3. **Strategy Search**
   - Beam search or greedy lookahead for short question sequences that drive |possibility set| → 1
   - Export human-readable strategy trees ("Ask Q1 → if yes: Q3, if no: Q2 …")

4. **Interactive Simulator**
   - ipywidgets "I am thinking of X station" mode that lets you play the hider while the notebook proposes questions

All of the above is already prototyped in earlier versions of this notebook; the current reset focuses on rock-solid data + visualization first (per audit learnings).

**Community resources** (excellent):
- https://taibeled.github.io/JetLagHideAndSeek/ (visual simulator + map)
- https://jetlag.neocities.org/ (community place lists for questions)

---
*Notebook initialized fresh following the original user request + lessons from AUDIT.md.*
